In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.5 MB/s eta 0:00:00


In [ ]:
import ultralytics
ultralytics.__version__

'8.4.155'

In [ ]:
import cv2
import pandas as pd
import os
import time

from ultralytics import YOLO
from tracker import *

model = YOLO("yolo26m.pt")

In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles:")
print(os.listdir())

Current directory: /content

Files:
['.config', '__pycache__', '.ipynb_checkpoints', 'tracker.py', 'yolo26m.pt', 'test_2.mp4', 'test_2_op.mp4', 'sample_data']


In [ ]:
class_list = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
              'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

In [ ]:
tracker=Tracker()
count=0

In [ ]:
cap=cv2.VideoCapture("content/test_2.mp4")

In [ ]:
down = {}
up = {}
counter_down = []
counter_up = []

red_line_y = 198
blue_line_y = 268
offset = 6

# Create a folder to save frames
if not os.path.exists('detected_frames'):
    os.makedirs('detected_frames')

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (1020, 500))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    count += 1
    # if count % 2 != 0:
    #     continue
    frame = cv2.resize(frame, (1020, 500))

    results = model.predict(frame)
    a = results[0].boxes.data
    a = a.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")
    list = []

    for index, row in px.iterrows():
        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])
        d = int(row[5])
        c = class_list[d]
        if 'car' in c:
            list.append([x1, y1, x2, y2])
    bbox_id = tracker.update(list)

    for bbox in bbox_id:
        x3, y3, x4, y4, id = bbox
        cx = int(x3 + x4) // 2
        cy = int(y3 + y4) // 2

        if red_line_y<(cy+offset) and red_line_y > (cy-offset):
           down[id]=time.time()   # current time when vehichle touch the first line
        if id in down:

           if blue_line_y<(cy+offset) and blue_line_y > (cy-offset):
             elapsed_time=time.time() - down[id]  # current time when vehicle touch the second line. Also we a re minusing the previous time ( current time of line 1)
             if counter_down.count(id)==0:
                counter_down.append(id)
                distance = 10 # meters
                a_speed_ms = distance / elapsed_time
                a_speed_kh = a_speed_ms * 3.6  # this will give kilometers per hour for each vehicle. This is the condition for going downside
                cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
                cv2.rectangle(frame, (x3, y3), (x4, y4), (0, 255, 0), 2)  # Draw bounding box
                cv2.putText(frame,str(id),(x3,y3),cv2.FONT_HERSHEY_COMPLEX,0.6,(255,255,255),1)
                cv2.putText(frame,str(int(a_speed_kh))+'Km/h',(x4,y4 ),cv2.FONT_HERSHEY_COMPLEX,0.8,(0,255,255),2)


        #####going UP blue line#####
        if blue_line_y<(cy+offset) and blue_line_y > (cy-offset):
           up[id]=time.time()
        if id in up:

           if red_line_y<(cy+offset) and red_line_y > (cy-offset):
             elapsed1_time=time.time() - up[id]
             # formula of speed= distance/time
             if counter_up.count(id)==0:
                counter_up.append(id)
                distance1 = 10 # meters  (Distance between the 2 lines is 10 meters )
                a_speed_ms1 = distance1 / elapsed1_time
                a_speed_kh1 = a_speed_ms1 * 3.6
                cv2.circle(frame,(cx,cy),4,(0,0,255),-1)
                cv2.rectangle(frame, (x3, y3), (x4, y4), (0, 255, 0), 2)  # Draw bounding box
                cv2.putText(frame,str(id),(x3,y3),cv2.FONT_HERSHEY_COMPLEX,0.6,(255,255,255),1)
                cv2.putText(frame,str(int(a_speed_kh1))+'Km/h',(x4,y4),cv2.FONT_HERSHEY_COMPLEX,0.8,(0,255,255),2)




    text_color = (0, 0, 0)  # Black color for text
    yellow_color = (0, 255, 255)  # Yellow color for background
    red_color = (0, 0, 255)  # Red color for lines
    blue_color = (255, 0, 0)  # Blue color for lines

    cv2.rectangle(frame, (0, 0), (250, 90), yellow_color, -1)

    cv2.line(frame, (172, 198), (774, 198), red_color, 2)
    cv2.putText(frame, ('Red Line'), (172, 198), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    cv2.line(frame, (8, 268), (927, 268), blue_color, 2)
    cv2.putText(frame, ('Blue Line'), (8, 268), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    cv2.putText(frame, ('Going Down - ' + str(len(counter_down))), (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)
    cv2.putText(frame, ('Going Up - ' + str(len(counter_up))), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_color, 1, cv2.LINE_AA)

    # Save frame
    frame_filename = f'detected_frames/frame_{count}.jpg'
    cv2.imwrite(frame_filename, frame)

    out.write(frame)

    # Don't use cv2.imshow() in Google Colab
    #if cv2.waitKey(0) & 0xFF == 27:
        #break

cap.release()
out.release()

print("Processing completed!")
print("Total Going Down:", len(counter_down))
print("Total Going Up:", len(counter_up))

Processing completed!
Total Going Down: 0
Total Going Up: 0


In [ ]:
import cv2
import pandas as pd
import os
import time

# Create tracker
tracker = Tracker()

# Frame counter
count = 0

# Open video
cap = cv2.VideoCapture("/content/test1.mp4")

if not cap.isOpened():
    raise Exception("Could not open test1.mp4")

# Get original FPS
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 20

print("Video FPS:", fps)

# Output video size
width = 1020
height = 500

# Create output folder
os.makedirs("/content/detected_frames", exist_ok=True)

# Video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    "/content/output.mp4",
    fourcc,
    fps,
    (width, height)
)

# Class names directly from YOLO model
class_list = model.names

print("Processing video...")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    count += 1

    # Resize frame
    frame = cv2.resize(frame, (width, height))

    # YOLO prediction
    results = model.predict(
        frame,
        verbose=False
    )

    # Get detections
    a = results[0].boxes.data

    # Convert tensor to NumPy
    a = a.detach().cpu().numpy()

    # Convert to DataFrame
    px = pd.DataFrame(a).astype("float")

    # Store car bounding boxes
    detections = []

    for index, row in px.iterrows():

        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])

        # Class ID
        d = int(row[5])

        # Class name
        c = class_list[d]

        # Only detect cars
        if c == "car":

            detections.append(
                [x1, y1, x2, y2]
            )

    # Update tracker
    bbox_id = tracker.update(detections)

    # Draw tracked vehicles
    for bbox in bbox_id:

        x3, y3, x4, y4, id = bbox

        # Center point
        cx = int((x3 + x4) / 2)
        cy = int((y3 + y4) / 2)

        # Draw center
        cv2.circle(
            frame,
            (cx, cy),
            4,
            (0, 0, 255),
            -1
        )

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x3, y3),
            (x4, y4),
            (0, 255, 0),
            2
        )

        # Draw tracking ID
        cv2.putText(
            frame,
            str(id),
            (cx, cy),
            cv2.FONT_HERSHEY_COMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    # ------------------------------------------------
    # LINE SETTINGS
    # ------------------------------------------------

    red_line_y = 198
    blue_line_y = 268
    offset = 7

    # Colors (B, G, R)
    text_color = (255, 255, 255)
    red_color = (0, 0, 255)
    blue_color = (255, 0, 0)

    # Red line
    cv2.line(
        frame,
        (172, red_line_y),
        (774, red_line_y),
        red_color,
        3
    )

    cv2.putText(
        frame,
        "Red Line",
        (172, red_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )

    # Blue line
    cv2.line(
        frame,
        (8, blue_line_y),
        (927, blue_line_y),
        blue_color,
        3
    )

    cv2.putText(
        frame,
        "Blue Line",
        (8, blue_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )

    # Save processed frame
    frame_filename = (
        f"/content/detected_frames/frame_{count}.jpg"
    )

    cv2.imwrite(
        frame_filename,
        frame
    )

    # Write frame to output video
    out.write(frame)

    # Progress
    if count % 50 == 0:
        print(f"Processed {count} frames")

# Release resources
cap.release()
out.release()

print("================================")
print("Video processing completed!")
print("Total frames:", count)
print("Output:", "/content/output.mp4")
print("================================")

Video FPS: 30.00413240194389
Processing video...
Processed 50 frames
Processed 100 frames
Processed 150 frames
Video processing completed!
Total frames: 169
Output: /content/output.mp4


In [ ]:
import cv2
import pandas as pd
import os
import time

from ultralytics import YOLO
from tracker import Tracker


# ============================================================
# LOAD YOLO26m
# ============================================================

model = YOLO("yolo26m.pt")

# Get class names directly from the model
class_list = model.names

print("Model loaded successfully!")


# ============================================================
# TRACKER
# ============================================================

tracker = Tracker()

count = 0


# ============================================================
# INPUT VIDEO
# ============================================================

video_path = "/content/highway_mini.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("ERROR: Could not open highway_mini.mp4")


# ============================================================
# VIDEO INFORMATION
# ============================================================

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 20

print("Video FPS:", fps)


# ============================================================
# OUTPUT VIDEO
# ============================================================

output_path = "/content/output.mp4"

width = 1020
height = 500

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# UP / DOWN DICTIONARIES
# ============================================================

down = {}
up = {}

counter_down = []
counter_up = []


# ============================================================
# PROCESS VIDEO
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    count += 1

    # Resize frame
    frame = cv2.resize(frame, (1020, 500))


    # ========================================================
    # YOLO PREDICTION
    # ========================================================

    results = model.predict(
        frame,
        verbose=False
    )


    # Get detection data
    a = results[0].boxes.data

    # Convert tensor to NumPy
    a = a.detach().cpu().numpy()

    # Convert to DataFrame
    px = pd.DataFrame(a).astype("float")


    # ========================================================
    # CAR DETECTIONS
    # ========================================================

    detections = []


    for index, row in px.iterrows():

        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])

        # Class ID
        d = int(row[5])

        # Class name
        c = class_list[d]


        # Only detect cars
        if c == "car":

            detections.append(
                [x1, y1, x2, y2]
            )


    # ========================================================
    # TRACK OBJECTS
    # ========================================================

    bbox_id = tracker.update(detections)


    # ========================================================
    # LINE SETTINGS
    # ========================================================

    red_line_y = 198
    blue_line_y = 268

    offset = 7


    # ========================================================
    # PROCESS TRACKED VEHICLES
    # ========================================================

    for bbox in bbox_id:

        x3, y3, x4, y4, id = bbox

        # Center point
        cx = int((x3 + x4) / 2)
        cy = int((y3 + y4) / 2)


        # ====================================================
        # RED LINE
        # ====================================================

        if red_line_y < (cy + offset) and red_line_y > (cy - offset):

            # Store time when vehicle touches red line
            down[id] = time.time()


            # =================================================
            # DRAW VEHICLE
            # =================================================

            cv2.circle(
                frame,
                (cx, cy),
                4,
                (0, 0, 255),
                -1
            )

            cv2.rectangle(
                frame,
                (x3, y3),
                (x4, y4),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                str(id),
                (cx, cy),
                cv2.FONT_HERSHEY_COMPLEX,
                0.8,
                (0, 255, 255),
                2
            )


            # =================================================
            # PRINT READABLE TIME
            # =================================================

            formatted_time = time.strftime(
                "%Y-%m-%d %H:%M:%S",
                time.localtime(down[id])
            )

            print(
                "Vehicle ID:",
                id,
                "| Red Line Time:",
                formatted_time
            )


    # ========================================================
    # DRAW RED LINE
    # ========================================================

    text_color = (255, 255, 255)

    red_color = (0, 0, 255)

    blue_color = (255, 0, 0)

    green_color = (0, 255, 0)


    cv2.line(
        frame,
        (172, 198),
        (774, 198),
        red_color,
        3
    )

    cv2.putText(
        frame,
        "Red Line",
        (172, 198),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # DRAW BLUE LINE
    # ========================================================

    cv2.line(
        frame,
        (8, 268),
        (927, 268),
        blue_color,
        3
    )

    cv2.putText(
        frame,
        "Blue Line",
        (8, 268),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # WRITE FRAME TO OUTPUT VIDEO
    # ========================================================

    out.write(frame)


    # ========================================================
    # PROGRESS
    # ========================================================

    if count % 50 == 0:

        print(
            "Processed frames:",
            count
        )


# ============================================================
# RELEASE VIDEO
# ============================================================

cap.release()
out.release()


print("\n====================================")
print("VIDEO PROCESSING COMPLETED")
print("====================================")
print("Total frames:", count)
print("Output:", output_path)
print("Vehicles touching red line:", len(down))
print("====================================")

Model loaded successfully!
Video FPS: 25.0
Vehicle ID: 2 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 2 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 2 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 1 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 4 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 0 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 4 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 0 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 4 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 0 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 4 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 0 | Red Line Time: 2026-09-05 19:29:42
Vehicle ID: 4 | Red Line Time: 2026-09-

In [ ]:
import cv2
import pandas as pd
import os
import time

from ultralytics import YOLO
from tracker import Tracker


# ============================================================
# YOLO26m MODEL
# ============================================================

model = YOLO("yolo26m.pt")

# Get class names directly from the model
class_list = model.names


# ============================================================
# TRACKER
# ============================================================

tracker = Tracker()

count = 0


# ============================================================
# VIDEO INPUT
# ============================================================

video_path = "/content/highway_mini.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Could not open highway_mini.mp4")


# ============================================================
# VIDEO SETTINGS
# ============================================================

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 20

width = 1020
height = 500


# ============================================================
# OUTPUT VIDEO
# ============================================================

output_path = "/content/output1.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# UP / DOWN
# ============================================================

down = {}
up = {}

counter_down = []
counter_up = []


# ============================================================
# LINE SETTINGS
# ============================================================

red_line_y = 198
blue_line_y = 268
offset = 7


print("Starting video processing...")


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    count += 1


    # --------------------------------------------------------
    # RESIZE FRAME
    # --------------------------------------------------------

    frame = cv2.resize(
        frame,
        (width, height)
    )


    # --------------------------------------------------------
    # YOLO26 DETECTION
    # --------------------------------------------------------

    results = model.predict(
        frame,
        verbose=False
    )


    # Get detection tensor
    a = results[0].boxes.data

    # Convert tensor to NumPy
    a = a.detach().cpu().numpy()

    # Convert to DataFrame
    px = pd.DataFrame(a).astype("float")


    # --------------------------------------------------------
    # CAR DETECTIONS
    # --------------------------------------------------------

    detections = []


    for index, row in px.iterrows():

        x1 = int(row[0])
        y1 = int(row[1])
        x2 = int(row[2])
        y2 = int(row[3])

        # Class ID
        d = int(row[5])

        # Class name
        c = class_list[d]


        # Only cars
        if c == "car":

            detections.append(
                [x1, y1, x2, y2]
            )


    # --------------------------------------------------------
    # TRACKING
    # --------------------------------------------------------

    bbox_id = tracker.update(detections)


    # --------------------------------------------------------
    # PROCESS TRACKED CARS
    # --------------------------------------------------------

    for bbox in bbox_id:

        x3, y3, x4, y4, id = bbox

        # Center point
        cx = int((x3 + x4) / 2)
        cy = int((y3 + y4) / 2)


        # ----------------------------------------------------
        # BLUE LINE CONDITION
        # ----------------------------------------------------

        if (
            blue_line_y < (cy + offset)
            and
            blue_line_y > (cy - offset)
        ):

            # Store time
            up[id] = time.time()


            # Draw center
            cv2.circle(
                frame,
                (cx, cy),
                4,
                (0, 0, 255),
                -1
            )


            # Draw ID
            cv2.putText(
                frame,
                str(id),
                (cx, cy),
                cv2.FONT_HERSHEY_COMPLEX,
                0.8,
                (0, 255, 255),
                2
            )


            # Add ID only once
            if id not in counter_up:

                counter_up.append(id)


    # --------------------------------------------------------
    # PRINT BLUE LINE DATA
    # --------------------------------------------------------

    if len(up) > 0:
        print("Blue line:", up)


    # --------------------------------------------------------
    # COLORS
    # --------------------------------------------------------

    text_color = (255, 255, 255)

    red_color = (0, 0, 255)

    blue_color = (255, 0, 0)

    green_color = (0, 255, 0)


    # --------------------------------------------------------
    # RED LINE
    # --------------------------------------------------------

    cv2.line(
        frame,
        (172, red_line_y),
        (774, red_line_y),
        red_color,
        3
    )

    cv2.putText(
        frame,
        "Red Line",
        (172, red_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # --------------------------------------------------------
    # BLUE LINE
    # --------------------------------------------------------

    cv2.line(
        frame,
        (8, blue_line_y),
        (927, blue_line_y),
        blue_color,
        3
    )

    cv2.putText(
        frame,
        "Blue Line",
        (8, blue_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # --------------------------------------------------------
    # WRITE OUTPUT FRAME
    # --------------------------------------------------------

    out.write(frame)


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if count % 50 == 0:

        print(
            "Processed frames:",
            count
        )


# ============================================================
# RELEASE
# ============================================================

cap.release()
out.release()


print()
print("======================================")
print("VIDEO PROCESSING COMPLETED")
print("======================================")
print("Total frames:", count)
print("Vehicles crossing blue line:", len(counter_up))
print("Output file:", output_path)
print("======================================")

Exception: Could not open highway_mini.mp4

**This show us result in cell**

In [ ]:
# ============================================================
# 1. INSTALL ULTRALYTICS
# ============================================================

!pip install -q ultralytics


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import cv2
import os
import pandas as pd

from ultralytics import YOLO
from tracker import Tracker

from IPython.display import Video, display


# ============================================================
# 3. LOAD YOLO26m MODEL
# ============================================================

model = YOLO("yolo26m.pt")

print("YOLO26m loaded successfully")


# ============================================================
# 4. VIDEO INPUT
# ============================================================

video_path = "/content/test_2.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(
        f"Could not open video: {video_path}"
    )

print("Video opened successfully")


# ============================================================
# 5. GET VIDEO INFORMATION
# ============================================================

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 20.0

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

video_width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

video_height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

video_duration = total_frames / fps


print("--------------------------------------")
print(f"Video FPS       : {fps:.2f}")
print(f"Total frames    : {total_frames}")
print(f"Original size   : {video_width} x {video_height}")
print(f"Duration        : {video_duration:.2f} seconds")
print("--------------------------------------")


# ============================================================
# 6. YOLO CLASS LIST
# ============================================================

class_list = model.names

print("YOLO Classes:")
print(class_list)


# ============================================================
# 7. CREATE TRACKER
# ============================================================

tracker = Tracker()


# ============================================================
# 8. VEHICLE CROSSING STORAGE
# ============================================================

# Vehicles that crossed RED line first
down = {}

# Vehicles that crossed BLUE line first
up = {}


# Completed vehicles
counter_down = []
counter_up = []


# ============================================================
# 9. SPEED STORAGE
# ============================================================

speed_down = {}
speed_up = {}


# ============================================================
# 10. LINE SETTINGS
# ============================================================

red_line_y = 198
blue_line_y = 268

offset = 6

# Physical distance between red and blue lines
DISTANCE_METERS = 10


# ============================================================
# 11. OUTPUT VIDEO
# ============================================================

output_path = "/content/test_2_op.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (1020, 500)
)

if not out.isOpened():
    raise ValueError(
        "Could not create output video"
    )

print(f"Output video will be saved to: {output_path}")


# ============================================================
# 12. FRAME COUNTER
# ============================================================

count = 0


# ============================================================
# 13. PROCESS VIDEO
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    count += 1


    # ========================================================
    # RESIZE FRAME
    # ========================================================

    frame = cv2.resize(
        frame,
        (1020, 500)
    )


    # ========================================================
    # YOLO DETECTION
    # ========================================================

    results = model.predict(
        frame,
        verbose=False
    )


    # ========================================================
    # GET DETECTION DATA
    # ========================================================

    a = results[0].boxes.data

    a = a.detach().cpu().numpy()

    px = pd.DataFrame(a).astype("float")


    # ========================================================
    # CAR DETECTIONS
    # ========================================================

    detections = []


    for index, row in px.iterrows():

        x1 = int(row[0])
        y1 = int(row[1])

        x2 = int(row[2])
        y2 = int(row[3])

        confidence = float(row[4])

        class_id = int(row[5])


        # Get class name
        c = class_list[class_id]


        # ====================================================
        # ONLY DETECT CARS
        # ====================================================

        if c == "car":

            detections.append(
                [x1, y1, x2, y2]
            )


    # ========================================================
    # UPDATE TRACKER
    # ========================================================

    bbox_id = tracker.update(
        detections
    )


    # ========================================================
    # PROCESS TRACKED VEHICLES
    # ========================================================

    for bbox in bbox_id:

        x3, y3, x4, y4, vehicle_id = bbox


        x3 = int(x3)
        y3 = int(y3)

        x4 = int(x4)
        y4 = int(y4)

        vehicle_id = int(vehicle_id)


        # ====================================================
        # CENTER POINT
        # ====================================================

        cx = int(
            (x3 + x4) / 2
        )

        cy = int(
            (y3 + y4) / 2
        )


        # ====================================================
        # CURRENT VIDEO TIME
        # ====================================================

        current_video_time = (
            count / fps
        )


        # ====================================================
        # UP DIRECTION
        #
        # BLUE LINE -> RED LINE
        # ====================================================

        if (
            blue_line_y < (cy + offset)
            and
            blue_line_y > (cy - offset)
        ):

            # Start UP timer only once
            if vehicle_id not in up:

                up[vehicle_id] = (
                    count,
                    current_video_time
                )

                print(
                    f"[Vehicle {vehicle_id}] "
                    f"BLUE line crossed "
                    f"(UP timer started)"
                )


        # ====================================================
        # CHECK RED LINE FOR UP VEHICLE
        # ====================================================

        if vehicle_id in up:

            if (
                red_line_y < (cy + offset)
                and
                red_line_y > (cy - offset)
            ):

                # Calculate only once
                if vehicle_id not in counter_up:

                    counter_up.append(
                        vehicle_id
                    )


                    # ========================================
                    # START INFORMATION
                    # ========================================

                    start_frame1, start_time1 = (
                        up[vehicle_id]
                    )


                    # ========================================
                    # END INFORMATION
                    # ========================================

                    end_frame1 = count

                    end_time1 = (
                        current_video_time
                    )


                    # ========================================
                    # ELAPSED TIME
                    # ========================================

                    elapsed1_time = (
                        end_time1 -
                        start_time1
                    )


                    # ========================================
                    # SPEED CALCULATION
                    # ========================================

                    if elapsed1_time > 0:

                        speed_ms1 = (
                            DISTANCE_METERS /
                            elapsed1_time
                        )

                        speed_kmh1 = (
                            speed_ms1 * 3.6
                        )


                        speed_up[
                            vehicle_id
                        ] = speed_kmh1


                        # ====================================
                        # PRINT UP SPEED
                        # ====================================

                        print(
                            "--------------------------------------"
                        )

                        print(
                            f"Vehicle ID : {vehicle_id}"
                        )

                        print(
                            "Direction  : UP"
                        )

                        print(
                            f"Start frame: {start_frame1}"
                        )

                        print(
                            f"End frame  : {end_frame1}"
                        )

                        print(
                            f"Elapsed    : "
                            f"{elapsed1_time:.2f} sec"
                        )

                        print(
                            f"Distance   : "
                            f"{DISTANCE_METERS} m"
                        )

                        print(
                            f"Speed      : "
                            f"{speed_kmh1:.2f} km/h"
                        )

                        print(
                            "--------------------------------------"
                        )


                        # ====================================
                        # DRAW VEHICLE
                        # ====================================

                        cv2.circle(
                            frame,
                            (cx, cy),
                            4,
                            (0, 0, 255),
                            -1
                        )


                        cv2.rectangle(
                            frame,
                            (x3, y3),
                            (x4, y4),
                            (0, 255, 0),
                            2
                        )


                        cv2.putText(
                            frame,
                            f"ID: {vehicle_id}",
                            (x3, y3),
                            cv2.FONT_HERSHEY_COMPLEX,
                            0.6,
                            (255, 255, 255),
                            1
                        )


                        cv2.putText(
                            frame,
                            f"{int(speed_kmh1)} Km/h",
                            (x3, y4),
                            cv2.FONT_HERSHEY_COMPLEX,
                            0.8,
                            (0, 255, 255),
                            2
                        )


        # ====================================================
        # DOWN DIRECTION
        #
        # RED LINE -> BLUE LINE
        # ====================================================

        if (
            red_line_y < (cy + offset)
            and
            red_line_y > (cy - offset)
        ):

            # Start DOWN timer only once
            if vehicle_id not in down:

                down[vehicle_id] = (
                    count,
                    current_video_time
                )

                print(
                    f"[Vehicle {vehicle_id}] "
                    f"RED line crossed "
                    f"(DOWN timer started)"
                )


        # ====================================================
        # CHECK BLUE LINE FOR DOWN VEHICLE
        # ====================================================

        if vehicle_id in down:

            if (
                blue_line_y < (cy + offset)
                and
                blue_line_y > (cy - offset)
            ):

                # Calculate only once
                if vehicle_id not in counter_down:

                    counter_down.append(
                        vehicle_id
                    )


                    # ========================================
                    # START INFORMATION
                    # ========================================

                    start_frame, start_time = (
                        down[vehicle_id]
                    )


                    # ========================================
                    # END INFORMATION
                    # ========================================

                    end_frame = count

                    end_time = (
                        current_video_time
                    )


                    # ========================================
                    # ELAPSED TIME
                    # ========================================

                    elapsed_time = (
                        end_time -
                        start_time
                    )


                    # ========================================
                    # SPEED CALCULATION
                    # ========================================

                    if elapsed_time > 0:

                        speed_ms = (
                            DISTANCE_METERS /
                            elapsed_time
                        )

                        speed_kmh = (
                            speed_ms * 3.6
                        )


                        speed_down[
                            vehicle_id
                        ] = speed_kmh


                        # ====================================
                        # PRINT DOWN SPEED
                        # ====================================

                        print(
                            "--------------------------------------"
                        )

                        print(
                            f"Vehicle ID : {vehicle_id}"
                        )

                        print(
                            "Direction  : DOWN"
                        )

                        print(
                            f"Start frame: {start_frame}"
                        )

                        print(
                            f"End frame  : {end_frame}"
                        )

                        print(
                            f"Elapsed    : "
                            f"{elapsed_time:.2f} sec"
                        )

                        print(
                            f"Distance   : "
                            f"{DISTANCE_METERS} m"
                        )

                        print(
                            f"Speed      : "
                            f"{speed_kmh:.2f} km/h"
                        )

                        print(
                            "--------------------------------------"
                        )


                        # ====================================
                        # DRAW VEHICLE
                        # ====================================

                        cv2.circle(
                            frame,
                            (cx, cy),
                            4,
                            (0, 0, 255),
                            -1
                        )


                        cv2.rectangle(
                            frame,
                            (x3, y3),
                            (x4, y4),
                            (0, 255, 0),
                            2
                        )


                        cv2.putText(
                            frame,
                            f"ID: {vehicle_id}",
                            (x3, y3),
                            cv2.FONT_HERSHEY_COMPLEX,
                            0.6,
                            (255, 255, 255),
                            1
                        )


                        cv2.putText(
                            frame,
                            f"{int(speed_kmh)} Km/h",
                            (x3, y4),
                            cv2.FONT_HERSHEY_COMPLEX,
                            0.8,
                            (0, 255, 255),
                            2
                        )


    # ========================================================
    # COLORS
    # ========================================================

    text_color = (0, 0, 0)

    yellow_color = (0, 255, 255)

    red_color = (0, 0, 255)

    blue_color = (255, 0, 0)


    # ========================================================
    # TOP INFORMATION BOX
    # ========================================================

    cv2.rectangle(
        frame,
        (0, 0),
        (250, 90),
        yellow_color,
        -1
    )


    # ========================================================
    # RED LINE
    # ========================================================

    cv2.line(
        frame,
        (172, red_line_y),
        (774, red_line_y),
        red_color,
        2
    )

    cv2.putText(
        frame,
        "Red Line",
        (172, red_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # BLUE LINE
    # ========================================================

    cv2.line(
        frame,
        (8, blue_line_y),
        (927, blue_line_y),
        blue_color,
        2
    )

    cv2.putText(
        frame,
        "Blue Line",
        (8, blue_line_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # UP COUNTER
    # ========================================================

    cv2.putText(
        frame,
        "Going Up - "
        + str(len(counter_up)),
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # DOWN COUNTER
    # ========================================================

    cv2.putText(
        frame,
        "Going Down - "
        + str(len(counter_down)),
        (10, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # WRITE PROCESSED VIDEO
    # ========================================================

    out.write(frame)


    # ========================================================
    # PROGRESS
    # ========================================================

    if count % 100 == 0:

        progress = (
            count / total_frames * 100
            if total_frames > 0
            else 0
        )

        print(
            f"Processing: "
            f"{count}/{total_frames} "
            f"({progress:.1f}%)"
        )


# ============================================================
# 14. RELEASE VIDEO
# ============================================================

cap.release()
out.release()


# ============================================================
# 15. FINAL RESULTS
# ============================================================

print("\n")
print("===================================================")
print("              PROCESSING COMPLETE")
print("===================================================")

print(
    f"Total frames processed : {count}"
)

print(
    f"Vehicles going UP      : "
    f"{len(counter_up)}"
)

print(
    f"Vehicles going DOWN    : "
    f"{len(counter_down)}"
)


# ============================================================
# 16. PRINT ALL SPEED RESULTS
# ============================================================

print("\n")
print("===================================================")
print("                 SPEED RESULTS")
print("===================================================")


# ============================================================
# UP SPEED RESULTS
# ============================================================

if len(speed_up) > 0:

    print("\nGOING UP:")

    for vehicle_id, speed in speed_up.items():

        print(
            f"Vehicle {vehicle_id} "
            f"-> {speed:.2f} km/h"
        )


# ============================================================
# DOWN SPEED RESULTS
# ============================================================

if len(speed_down) > 0:

    print("\nGOING DOWN:")

    for vehicle_id, speed in speed_down.items():

        print(
            f"Vehicle {vehicle_id} "
            f"-> {speed:.2f} km/h"
        )


# ============================================================
# NO COMPLETED VEHICLES
# ============================================================

if len(speed_down) == 0 and len(speed_up) == 0:

    print(
        "\nNo vehicles completed both line crossings."
    )


# ============================================================
# OUTPUT LOCATION
# ============================================================

print("\n")
print("===================================================")
print("OUTPUT VIDEO")
print("===================================================")

print(
    f"Saved to: {output_path}"
)


# ============================================================
# DISPLAY OUTPUT VIDEO IN COLAB
# ============================================================

display(
    Video(
        output_path,
        embed=True
    )
)

Output hidden; open in https://colab.research.google.com to view.